<a href="https://colab.research.google.com/github/MuhammadAhmed196/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadAhmed196/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---------------------------------------------------------
# SECTION 1 — BUILD THE FEATURE VECTOR
# ---------------------------------------------------------

import pandas as pd
import numpy as np

# Public starter dataset used for the feature/leakage audit
data_url = (
    "https://raw.githubusercontent.com/"
    "flyrank-bih/flyrank-ml-internship-starter/"
    "main/data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(data_url)

print("Rows:", len(df))
print("Columns:", len(df.columns))

# Final decision-time feature set used by the capstone model
feature_columns = [
    "impressions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "word_count"
]

# Keep only features that are actually present
available_features = [
    col for col in feature_columns
    if col in df.columns
]

feature_frame = df[available_features].copy()

print("\nFinal feature columns:")
for col in available_features:
    print("-", col)

print("\nFeature-frame shape:", feature_frame.shape)

print("\nMissing values:")
display(
    feature_frame.isna()
    .sum()
    .to_frame("missing_count")
)

print("\nFeature vector build check: PASS")


Rows: 30000
Columns: 44

Final feature columns:
- impressions_90d
- avg_position
- ctr
- content_age_days
- days_since_last_update
- word_count

Feature-frame shape: (30000, 6)

Missing values:


,missing_count
impressions_90d,0
avg_position,0
ctr,0
content_age_days,0
days_since_last_update,0
word_count,7699



Feature vector build check: PASS


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---------------------------------------------------------
# SECTION 2 — FEATURE NOTES
# ---------------------------------------------------------

feature_notes = pd.DataFrame({
    "feature": [
        "impressions_90d",
        "avg_position",
        "ctr",
        "content_age_days",
        "days_since_last_update",
        "word_count"
    ],
    "meaning": [
        "Search visibility measured by impressions in the recent 90-day window.",
        "Average search position observed for the page.",
        "Click-through rate derived from observed impressions and clicks.",
        "Approximate age of the content page.",
        "Days since the last recorded content update.",
        "Stored page-level word count."
    ],
    "missing_handling": [
        "Median imputation in the modelling pipeline.",
        "Median imputation in the modelling pipeline.",
        "Median imputation in the modelling pipeline.",
        "Median imputation in the modelling pipeline.",
        "Median imputation in the modelling pipeline.",
        "Median imputation in the modelling pipeline."
    ],
    "available_before_prediction": [
        True,
        True,
        True,
        True,
        True,
        True
    ]
})

display(feature_notes)

print("\nFeature availability check:")
print(
    "All final predictive features are defined as pre-decision "
    "page-level information."
)
print("Feature notes check: PASS")


,feature,meaning,missing_handling,available_before_prediction
0,impressions_90d,Search visibility measured by impressions in t...,Median imputation in the modelling pipeline.,True
1,avg_position,Average search position observed for the page.,Median imputation in the modelling pipeline.,True
2,ctr,Click-through rate derived from observed impre...,Median imputation in the modelling pipeline.,True
3,content_age_days,Approximate age of the content page.,Median imputation in the modelling pipeline.,True
4,days_since_last_update,Days since the last recorded content update.,Median imputation in the modelling pipeline.,True
5,word_count,Stored page-level word count.,Median imputation in the modelling pipeline.,True



Feature availability check:
All final predictive features are defined as pre-decision page-level information.
Feature notes check: PASS


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---------------------------------------------------------
# SECTION 3 — LEAKAGE HUNT
# ---------------------------------------------------------

leakage_prone_fields = [
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "future_decline_answer",
    "product_flag",
    "needs_refresh",
    "needs_ctr_fix",
    "quick_win"
]

future_or_label_fields_present = [
    col for col in leakage_prone_fields
    if col in df.columns
]

used_as_features = [
    col for col in available_features
    if col in leakage_prone_fields
]

print("Leakage-prone fields present in dataset:")
for col in future_or_label_fields_present:
    print("-", col)

print("\nLeakage-prone fields used as features:")
print(used_as_features)

print("\nFeature leakage check:",
      "PASS" if len(used_as_features) == 0 else "FAIL")

print(
    "\nFuture-window / label-derived fields included in final "
    "feature vector:",
    "NO" if len(used_as_features) == 0 else "YES"
)


Leakage-prone fields present in dataset:
- trend_direction
- trend_pct

Leakage-prone fields used as features:
[]

Feature leakage check: PASS

Future-window / label-derived fields included in final feature vector: NO


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---------------------------------------------------------
# SECTION 4 — WHAT I EXCLUDED AND WHY
# ---------------------------------------------------------

excluded_fields = pd.DataFrame({
    "field": [
        "client identifiers",
        "content identifiers",
        "is_declining_label",
        "trend_direction",
        "trend_pct",
        "future_decline_answer",
        "product_flag",
        "needs_refresh",
        "needs_ctr_fix",
        "quick_win"
    ],
    "reason": [
        "Used for grouping/joining, not generalizable predictive behavior.",
        "Identifies a content record rather than a generalizable signal.",
        "Evaluation target; using it as an input would directly leak the answer.",
        "Directly describes the observed trend and can encode the target relationship.",
        "Derived from outcome/trend information and therefore leakage-prone.",
        "Future answer derived from the target; direct leakage.",
        "Product-decision information not intended as a predictive feature.",
        "Existing decision output; using it would duplicate the decision rule.",
        "Existing product decision output; not an independent predictive signal.",
        "Existing product decision output; not an independent predictive signal."
    ]
})

display(excluded_fields)

print("\nExcluded-field audit: PASS")
print("Final predictive feature count:", len(available_features))
print("Final leakage-prone feature count:", len(used_as_features))


,field,reason
0,client identifiers,"Used for grouping/joining, not generalizable p..."
1,content identifiers,Identifies a content record rather than a gene...
2,is_declining_label,Evaluation target; using it as an input would ...
3,trend_direction,Directly describes the observed trend and can ...
4,trend_pct,Derived from outcome/trend information and the...
5,future_decline_answer,Future answer derived from the target; direct ...
6,product_flag,Product-decision information not intended as a...
7,needs_refresh,Existing decision output; using it would dupli...
8,needs_ctr_fix,Existing product decision output; not an indep...
9,quick_win,Existing product decision output; not an indep...



Excluded-field audit: PASS
Final predictive feature count: 6
Final leakage-prone feature count: 0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.